# Run Coding Assistant — Public Network

This notebook demonstrates the coding assistant experience with **all 5 MCP servers** active, leveraging both external services (internet) and internal tools.

**Environment**: Public network (internet access available)

| Tool | Server | Use Case |
|------|--------|----------|
| Context7 | External | Official library documentation (FastAPI, SQLAlchemy, etc.) |
| DuckDuckGo | External | Web search for best practices, StackOverflow, blogs |
| Code Sandbox | Local | Execute and verify code snippets |
| Codebase Search | Local AI | Semantic search over internal codebase |
| Repo Docs | Local AI | Q&A over internal architecture/runbook docs |

> **Compare with**: `3_run_closed_coding_assistant.ipynb` — same scenario with only air-gapped tools.

**Scenario**: Add a "daily specials" feature to the `cafe-order-system`.

## 1. Verify MCP Server Connectivity

In [1]:
import subprocess, json

result = subprocess.run(
    ["oc", "get", "routes", "-n", "mcp-servers",
     "-o", "jsonpath={range .items[*]}{.metadata.name}={.spec.host}\n{end}"],
    capture_output=True, text=True
)

ROUTES = {}
for line in result.stdout.strip().split("\n"):
    if "=" in line:
        name, host = line.split("=", 1)
        ROUTES[name] = f"https://{host}/mcp"

EXPECTED = ["mcp-context7", "mcp-duckduckgo", "mcp-code-sandbox", "mcp-codebase-search", "mcp-repo-docs"]

print("MCP Servers (Public Network Mode — all 5 active):")
print("=" * 65)
for name in EXPECTED:
    url = ROUTES.get(name, "NOT DEPLOYED")
    status = "READY" if name in ROUTES else "MISSING"
    airgap = "Internet" if name in ("mcp-context7", "mcp-duckduckgo") else "Local"
    print(f"  [{status}] {name:<25} ({airgap})  {url}")

missing = [n for n in EXPECTED if n not in ROUTES]
if missing:
    print(f"\n  WARNING: Missing servers: {missing}")
    print("  Run 1_mcp_servers/2_deploy_mcp_servers.ipynb first.")
else:
    print(f"\n  All 5 servers ready.")

MCP Servers (Public Network Mode — all 5 active):
  [READY] mcp-context7              (Internet)  https://mcp-context7-mcp-servers.apps.openshift-cluster.sandbox1785.opentlc.com/mcp
  [READY] mcp-duckduckgo            (Internet)  https://mcp-duckduckgo-mcp-servers.apps.openshift-cluster.sandbox1785.opentlc.com/mcp
  [READY] mcp-code-sandbox          (Local)  https://mcp-code-sandbox-mcp-servers.apps.openshift-cluster.sandbox1785.opentlc.com/mcp
  [READY] mcp-codebase-search       (Local)  https://mcp-codebase-search-mcp-servers.apps.openshift-cluster.sandbox1785.opentlc.com/mcp
  [READY] mcp-repo-docs             (Local)  https://mcp-repo-docs-mcp-servers.apps.openshift-cluster.sandbox1785.opentlc.com/mcp

  All 5 servers ready.


## 2. Generate IDE Configuration (All Servers)

Generate MCP config with all 5 MCP servers enabled for your IDE.

In [2]:
cursor_config = {"mcpServers": {}}
opencode_config = {"$schema": "https://opencode.ai/config.json", "mcp": {}}

for name, url in ROUTES.items():
    short = name.replace("mcp-", "")
    cursor_config["mcpServers"][short] = {"url": url}
    opencode_config["mcp"][short] = {"type": "remote", "url": url}

print("=== .cursor/mcp.json (PUBLIC — all servers) ===")
print(json.dumps(cursor_config, indent=2))

print("")
print("=== opencode.json (PUBLIC — all servers) ===")
print(json.dumps(opencode_config, indent=2))

print("\nCopy the appropriate config to your project root.")
print("All team members get the same MCP tools automatically.")

=== .cursor/mcp.json (PUBLIC — all 5 servers) ===
{
  "mcpServers": {
    "code-sandbox": {
      "url": "https://mcp-code-sandbox-mcp-servers.apps.openshift-cluster.sandbox1785.opentlc.com/mcp"
    },
    "codebase-search": {
      "url": "https://mcp-codebase-search-mcp-servers.apps.openshift-cluster.sandbox1785.opentlc.com/mcp"
    },
    "context7": {
      "url": "https://mcp-context7-mcp-servers.apps.openshift-cluster.sandbox1785.opentlc.com/mcp"
    },
    "duckduckgo": {
      "url": "https://mcp-duckduckgo-mcp-servers.apps.openshift-cluster.sandbox1785.opentlc.com/mcp"
    },
    "repo-docs": {
      "url": "https://mcp-repo-docs-mcp-servers.apps.openshift-cluster.sandbox1785.opentlc.com/mcp"
    },
    "ocp-server": {
      "url": "https://ocp-mcp-server-mcp-servers.apps.openshift-cluster.sandbox1785.opentlc.com/mcp"
    }
  }
}

Copy this to your project's .cursor/mcp.json
All team members get the same MCP tools automatically.


## 3. Scenario: Add "Daily Specials" to cafe-order-system

We'll walk through how the coding assistant uses **all 5 tools** together to implement a new feature.

### Task
Add a `/api/specials` endpoint that returns today's daily special menu items (discounted items selected by the barista each morning).

---

### Step 1: Understand the existing codebase

**Tool: Codebase Search** — "How are menu items structured?"

In [3]:
# Simulate what the agent does: search internal codebase for menu item structure
import subprocess, json

init = json.dumps({"jsonrpc":"2.0","id":1,"method":"initialize","params":{"protocolVersion":"2025-03-26","capabilities":{},"clientInfo":{"name":"test","version":"1.0"}}})
call = json.dumps({"jsonrpc":"2.0","id":2,"method":"tools/call","params":{"name":"search_code","arguments":{"query":"menu item model class definition","top_k":2}}})

url = ROUTES.get("mcp-codebase-search", "")
if url:
    # Initialize
    r = subprocess.run(["curl","-sk","-X","POST","-H","Content-Type: application/json","-H","Accept: application/json, text/event-stream","-d",init,"-m","10",url], capture_output=True, text=True)
    # Call tool
    r = subprocess.run(["curl","-sk","-X","POST","-H","Content-Type: application/json","-H","Accept: application/json, text/event-stream","-d",call,"-m","15",url], capture_output=True, text=True)
    print("Agent uses: Codebase Search > search_code")
    print("Query: 'menu item model class definition'")
    print("=" * 60)
    # Parse SSE response
    for line in r.stdout.split("\n"):
        if line.startswith("data:"):
            try:
                data = json.loads(line[5:])
                if "result" in data:
                    for content in data["result"].get("content", []):
                        print(content.get("text", "")[:800])
            except: pass
else:
    print("mcp-codebase-search not available")

Agent uses: Codebase Search > search_code
Query: 'menu item model class definition'
--- Result 1 (score: 0.550) ---
File: models.py (lines 71-83)



class OrderItem(Base):
    __tablename__ = "order_items"

    id = Column(Integer, primary_key=True, index=True)
    order_id = Column(Integer, ForeignKey("orders.id"), nullable=False)
    menu_item_id = Column(Integer, ForeignKey("menu_items.id"), nullable=False)
    quantity = Column(Integer, default=1)
    customization = Column(String(200))

    order = relationship("Order", back_populates="items")
    menu_item = relationship("MenuItem", back_populates="order_items")

--- Result 2 (score: 0.497) ---
File: ..2026_06_18_08_45_36.1191552457/schemas.py (lines 1-40)

from datetime import datetime
from typing import Optional

from pydantic import BaseModel, Field

from app.models import MenuCategory, OrderStatus


class MenuIt


### Step 2: Look up FastAPI documentation

**Tool: Context7** — "How to create a FastAPI endpoint with query parameters?"

Context7 fetches the **latest official FastAPI docs** — something only possible with internet access.

In [4]:
url = ROUTES.get("mcp-context7", "")
if url:
    call = json.dumps({"jsonrpc":"2.0","id":2,"method":"tools/call","params":{"name":"resolve-library-id","arguments":{"libraryName":"fastapi"}}})
    r = subprocess.run(["curl","-sk","-X","POST","-H","Content-Type: application/json","-H","Accept: application/json, text/event-stream","-d",init,"-m","10",url], capture_output=True, text=True)
    r = subprocess.run(["curl","-sk","-X","POST","-H","Content-Type: application/json","-H","Accept: application/json, text/event-stream","-d",call,"-m","15",url], capture_output=True, text=True)
    print("Agent uses: Context7 > resolve-library-id")
    print("Query: 'fastapi'")
    print("=" * 60)
    for line in r.stdout.split("\n"):
        if line.startswith("data:"):
            try:
                data = json.loads(line[5:])
                if "result" in data:
                    for content in data["result"].get("content", []):
                        print(content.get("text", "")[:500])
            except: pass
else:
    print("mcp-context7 not available (requires internet)")

Agent uses: Context7 > resolve-library-id
Query: 'fastapi'
MCP error -32602: Input validation error: Invalid arguments for tool resolve-library-id: [
  {
    "expected": "string",
    "code": "invalid_type",
    "path": [
      "query"
    ],
    "message": "Invalid input: expected string, received undefined"
  }
]


### Step 3: Search web for best practices

**Tool: DuckDuckGo** — "FastAPI daily scheduler pattern SQLAlchemy"

DuckDuckGo searches the web for relevant blog posts, StackOverflow answers, and tutorials.

In [5]:
url = ROUTES.get("mcp-duckduckgo", "")
if url:
    call = json.dumps({"jsonrpc":"2.0","id":2,"method":"tools/call","params":{"name":"duckduckgo_search","arguments":{"query":"FastAPI daily scheduled task SQLAlchemy best practice","count":3}}})
    r = subprocess.run(["curl","-sk","-X","POST","-H","Content-Type: application/json","-H","Accept: application/json, text/event-stream","-d",init,"-m","10",url], capture_output=True, text=True)
    r = subprocess.run(["curl","-sk","-X","POST","-H","Content-Type: application/json","-H","Accept: application/json, text/event-stream","-d",call,"-m","15",url], capture_output=True, text=True)
    print("Agent uses: DuckDuckGo > duckduckgo_search")
    print("Query: 'FastAPI daily scheduled task SQLAlchemy best practice'")
    print("=" * 60)
    for line in r.stdout.split("\n"):
        if line.startswith("data:"):
            try:
                data = json.loads(line[5:])
                if "result" in data:
                    for content in data["result"].get("content", []):
                        print(content.get("text", "")[:600])
            except: pass
else:
    print("mcp-duckduckgo not available (requires internet)")

Agent uses: DuckDuckGo > duckduckgo_search
Query: 'FastAPI daily scheduled task SQLAlchemy best practice'
Unknown tool: duckduckgo_search


### Step 4: Check internal documentation

**Tool: Repo Docs** — "What is the API endpoint pattern in this project?"

In [6]:
url = ROUTES.get("mcp-repo-docs", "")
if url:
    call = json.dumps({"jsonrpc":"2.0","id":2,"method":"tools/call","params":{"name":"search_docs","arguments":{"query":"API endpoint design pattern and response format","top_k":2}}})
    r = subprocess.run(["curl","-sk","-X","POST","-H","Content-Type: application/json","-H","Accept: application/json, text/event-stream","-d",init,"-m","10",url], capture_output=True, text=True)
    r = subprocess.run(["curl","-sk","-X","POST","-H","Content-Type: application/json","-H","Accept: application/json, text/event-stream","-d",call,"-m","15",url], capture_output=True, text=True)
    print("Agent uses: Repo Docs > search_docs")
    print("Query: 'API endpoint design pattern and response format'")
    print("=" * 60)
    for line in r.stdout.split("\n"):
        if line.startswith("data:"):
            try:
                data = json.loads(line[5:])
                if "result" in data:
                    for content in data["result"].get("content", []):
                        print(content.get("text", "")[:600])
            except: pass
else:
    print("mcp-repo-docs not available")

Agent uses: Repo Docs > search_docs
Query: 'API endpoint design pattern and response format'
--- Result 1 (score: 0.384) ---
Source: ..2026_06_17_08_53_51.704143465/api-guide.md > Health Check

```
GET /health
```

Response:
```json
{"status": "healthy", "service": "Cafe Order System", "version": "1.2.0"}
```

---

--- Result 2 (score: 0.377) ---
Source: api-guide.md > Health Check

```
GET /health
```

Response:
```json
{"status": "healthy", "service": "Cafe Order System", "version": "1.2.0"}
```

---


### Step 5: Execute and verify the implementation

**Tool: Code Sandbox** — Run the generated code to verify it works.

In [7]:
url = ROUTES.get("mcp-code-sandbox", "")
if url:
    test_code = '''import json\nspecials = [{"name": "아메리카노", "price": 3500, "original_price": 4500, "discount": "22%"},\n            {"name": "크루아상", "price": 3000, "original_price": 4000, "discount": "25%"}]\nprint(json.dumps({"date": "2026-06-16", "specials": specials}, ensure_ascii=False, indent=2))'''
    call = json.dumps({"jsonrpc":"2.0","id":2,"method":"tools/call","params":{"name":"execute_code","arguments":{"code":test_code,"language":"python"}}})
    r = subprocess.run(["curl","-sk","-X","POST","-H","Content-Type: application/json","-H","Accept: application/json, text/event-stream","-d",init,"-m","10",url], capture_output=True, text=True)
    r = subprocess.run(["curl","-sk","-X","POST","-H","Content-Type: application/json","-H","Accept: application/json, text/event-stream","-d",call,"-m","15",url], capture_output=True, text=True)
    print("Agent uses: Code Sandbox > execute_code")
    print("Action: Run generated daily specials API response")
    print("=" * 60)
    for line in r.stdout.split("\n"):
        if line.startswith("data:"):
            try:
                data = json.loads(line[5:])
                if "result" in data:
                    for content in data["result"].get("content", []):
                        print(content.get("text", ""))
            except: pass
else:
    print("mcp-code-sandbox not available")

Agent uses: Code Sandbox > execute_code
Action: Run generated daily specials API response
[python] OK (0.02s)

stdout:
{
  "date": "2026-06-16",
  "specials": [
    {
      "name": "아메리카노",
      "price": 3500,
      "original_price": 4500,
      "discount": "22%"
    },
    {
      "name": "크루아상",
      "price": 3000,
      "original_price": 4000,
      "discount": "25%"
    }
  ]
}


## 4. Summary: Public Network Tool Usage

| Step | Task | Tool Used | Source |
|------|------|-----------|--------|
| 1 | Understand existing code | **Codebase Search** | Internal codebase (local) |
| 2 | Look up library docs | **Context7** | Official FastAPI docs (internet) |
| 3 | Search best practices | **DuckDuckGo** | Web search (internet) |
| 4 | Check internal conventions | **Repo Docs** | Architecture/API docs (local) |
| 5 | Verify implementation | **Code Sandbox** | Local execution (local) |

### Key advantage of Public mode:
- Access to **latest official documentation** via Context7
- Access to **community knowledge** (blogs, SO) via DuckDuckGo
- Combined with internal tools for project-specific context

## Next Steps

- `3_run_closed_coding_assistant.ipynb` — See how the same task works in an air-gapped environment
- `../2_maas/` — Add MaaS gateway for auth and rate limiting on top of these tools